# 00 - Environment and data check

This notebook is an *internal check* notebook for the Wind Turbine Hub Height project.

It is **not** the main project documentation - the README.md is.
Use this notebook to quickly confirm that:

- the Python environment is set up correctly (all required packages installed)
- the project folder structure exists
- the required datasets (HOSTRADA, LBM-DE2021, Germany border shapefile) are downloaded and have the expected structure

## Environment check

In [1]:
import sys
print('Python version:', sys.version)

Python version: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]


In [2]:
# Check that all required packages are installed and importable.
# If any of these fail, run: pip install -r requirements.txt

packages = [
    'numpy',
    'xarray',
    'dask',
    'netCDF4',
    'geopandas',
    'shapely',
    'pyproj',
    'requests',
    'matplotlib',
]

for package_name in packages:
    try:
        module = __import__(package_name)
        version = getattr(module, '__version__', 'unknown version')
        print(f'OK       {package_name:<12} ({version})')
    except ImportError:
        print(f'MISSING  {package_name:<12} - run: pip install {package_name}')

OK       numpy        (2.4.4)
OK       xarray       (2026.7.0)
OK       dask         (2026.8.0)
OK       netCDF4      (1.7.4)
OK       geopandas    (1.1.4)
OK       shapely      (2.1.2)
OK       pyproj       (3.7.2)
OK       requests     (2.33.1)
OK       matplotlib   (3.10.9)


## Folder and file check

In [3]:
from pathlib import Path

cwd = Path.cwd()
print('Current working directory:', cwd)

# The notebook lives in notebooks/, so the project root is one level up.
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
print('Project root             :', project_root)

Current working directory: c:\Users\Lenovo\OneDrive\Dokumente\ISU\6.1 Virtual reality\wind-turbine-height\notebooks
Project root             : c:\Users\Lenovo\OneDrive\Dokumente\ISU\6.1 Virtual reality\wind-turbine-height


In [4]:
# Check that the main project folders and source files exist.
expected = [
    'data/raw',
    'data/raw/hostrada',
    'data/raw/LBM',
    'data/raw/ShapeGermany',
    'data/processed',
    'notebooks',
    'src',
    'src/config.py',
    'src/coordinates.py',
    'src/border_download.py',
    'src/hostrada_download.py',
    'src/hostrada_prep.py',
    'src/lbm_download.py',
    'src/lbm_prep.py',
    'src/alpha_calculation.py',
    'src/wind_profile.py',
    'src/visualization.py',
    'main.py',
    'README.md',
    'requirements.txt',
]

for item in expected:
    path = project_root / item
    marker = 'OK ' if path.exists() else 'MISSING'
    print(f'{marker}  {item}')

OK   data/raw
OK   data/raw/hostrada
OK   data/raw/LBM
OK   data/raw/ShapeGermany
OK   data/processed
OK   notebooks
OK   src
OK   src/config.py
OK   src/coordinates.py
OK   src/border_download.py
OK   src/hostrada_download.py
OK   src/hostrada_prep.py
OK   src/lbm_download.py
OK   src/lbm_prep.py
OK   src/alpha_calculation.py
OK   src/wind_profile.py
OK   src/visualization.py
OK   main.py
OK   README.md
OK   requirements.txt


## HOSTRADA data check

Confirms that HOSTRADA wind speed files have been downloaded (see `src/hostrada_download.py`) and that at least one file can be opened correctly with the expected variables.

In [5]:
hostrada_dir = project_root / 'data' / 'raw' / 'hostrada'
hostrada_files = sorted(hostrada_dir.glob('*.nc')) if hostrada_dir.exists() else []

print(f'HOSTRADA files found: {len(hostrada_files)}')
if len(hostrada_files) == 0:
    print('MISSING - run: python src/hostrada_download.py')
else:
    print('First file:', hostrada_files[0].name)
    print('Last file :', hostrada_files[-1].name)

HOSTRADA files found: 120
First file: sfcWind_1hr_HOSTRADA-v1-0_BE_gn_2016010100-2016013123.nc
Last file : sfcWind_1hr_HOSTRADA-v1-0_BE_gn_2025120100-2025123123.nc


In [6]:
# Open one HOSTRADA file and confirm it has the expected variable and coordinates.
import xarray as xr

if hostrada_files:
    ds = xr.open_dataset(hostrada_files[0])

    expected_vars = ['sfcWind']
    expected_coords = ['time', 'lat', 'lon']

    for var in expected_vars:
        marker = 'OK ' if var in ds.data_vars else 'MISSING'
        print(f'{marker}  data variable: {var}')

    for coord in expected_coords:
        marker = 'OK ' if coord in ds.coords else 'MISSING'
        print(f'{marker}  coordinate: {coord}')

    ds.close()
else:
    print('Skipped - no HOSTRADA files found.')

OK   data variable: sfcWind
OK   coordinate: time
OK   coordinate: lat
OK   coordinate: lon


## LBM-DE2021 land cover data check

Confirms the LBM-DE2021 geopackage exists and has the expected layer and CLC code column. Only a few rows are read to keep this check fast (the full file is several GB).

In [7]:
lbm_path = project_root / 'data' / 'raw' / 'LBM' / 'LBMDE.gpkg'
print('LBM-DE file found:', lbm_path.exists())

if not lbm_path.exists():
    print('MISSING - run: python src/lbm_download.py')

LBM-DE file found: True


In [8]:
import geopandas as gpd

if lbm_path.exists():
    layers = gpd.list_layers(lbm_path)
    print('Layers found:')
    print(layers)

    sample = gpd.read_file(lbm_path, layer='LBMDE_2021', rows=5)

    marker = 'OK ' if 'CLC21' in sample.columns else 'MISSING'
    print(f'{marker}  CLC21 column present')

    print('CRS:', sample.crs)
else:
    print('Skipped - LBM-DE file not found.')

Layers found:
         name geometry_type
0  LBMDE_2021  MultiPolygon
OK   CLC21 column present
CRS: EPSG:25832


## Germany border shapefile check

Confirms the Germany border shapefile (used in `src/coordinates.py` for precise coordinate validation) exists and contains exactly one country row.

In [9]:
border_path = project_root / 'data' / 'raw' / 'ShapeGermany' / 'Germany.shp'
print('Germany border shapefile found:', border_path.exists())

if border_path.exists():
    world_borders = gpd.read_file(border_path)
    germany = world_borders[world_borders['SOVEREIGNT'] == 'Germany']

    print('Rows matching "Germany":', len(germany))
    print('CRS:', germany.crs)
else:
    print('MISSING - run: python src/border_download.py')

Germany border shapefile found: True
Rows matching "Germany": 1
CRS: EPSG:4326


## Quick pipeline test

Runs the coordinate validation step only (fast) to confirm the source code itself is importable and working, without triggering the slow HOSTRADA data loading.

In [10]:
import sys
sys.path.insert(0, str(project_root / 'src'))

from coordinates import validate_coordinate, CoordinateOutOfBoundsError

try:
    coord = validate_coordinate(52.5200, 13.4050)  # Berlin
    print('OK  coordinate validation works:', coord)
except Exception as e:
    print('FAILED:', e)

OK  coordinate validation works: Coordinate(lat=52.52, lon=13.405)


If all checks above are OK, the environment and the datasets are ready.

Next step: enter the coordinates of your location in the main.py file and run

```
python main.py
```

Note: the first full run will take some time, since it opens all configured HOSTRADA files (one per month, see `HOSTRADA_YEARS` in `src/config.py`).